# LAB 10 — QA Fine-tuning & Evaluation

Notebook này thực hiện:
1. Load SQuAD dataset
2. Baseline (keyword overlap)
3. Fine-tune pretrained Transformer cho extractive QA
4. Experiments: context length, EM vs F1, answer length, question/context matching
5. Error analysis
6. Visualization

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
import random
import time

from qa_model import (
    SQuADDataset, ExtractiveQAModel, KeywordOverlapBaseline,
    compute_exact_match, compute_f1, evaluate_predictions,
    train_epoch, evaluate_model, predict_answers, normalize_answer
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load SQuAD Dataset

In [ ]:
# Load SQuAD v1.1
try:
    dataset = load_dataset("rajpurkar/squad", trust_remote_code=True)
except Exception:
    # Fallback: download JSON files directly
    import json, urllib.request, os
    os.makedirs('squad_data', exist_ok=True)
    for split, url in [('train', 'https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json'),
                       ('dev', 'https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json')]:
        fpath = f'squad_data/{split}.json'
        if not os.path.exists(fpath):
            print(f'Downloading {split}...'); urllib.request.urlretrieve(url, fpath)
    def parse_squad(path):
        with open(path) as f: raw = json.load(f)['data']
        rows = []
        for article in raw:
            for para in article['paragraphs']:
                for qa in para['qas']:
                    rows.append({'context': para['context'], 'question': qa['question'],
                                 'answers': {'text': [a['text'] for a in qa['answers']],
                                             'answer_start': [a['answer_start'] for a in qa['answers']]},
                                 'id': qa['id']})
        return rows
    from datasets import Dataset as HFDataset
    dataset = {'train': HFDataset.from_list(parse_squad('squad_data/train.json')),
               'validation': HFDataset.from_list(parse_squad('squad_data/dev.json'))}
    print('Loaded SQuAD from local JSON files.')

print(f"Train: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")

# Lấy subset để training nhanh hơn (có thể tăng nếu có GPU mạnh)
TRAIN_SIZE = 5000
VAL_SIZE = 1000

train_data = dataset['train'].select(range(TRAIN_SIZE))
val_data = dataset['validation'].select(range(VAL_SIZE))

# Xem sample
sample = train_data[0]
print(f"\n=== Sample ===")
print(f"Context: {sample['context'][:200]}...")
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']['text']}")
print(f"Answer start: {sample['answers']['answer_start']}")

## 2. Baseline — Keyword Overlap

In [ ]:
baseline = KeywordOverlapBaseline()

baseline_preds = []
baseline_golds = []

for item in val_data:
    pred = baseline.predict(item['question'], item['context'])
    gold = item['answers']['text'][0]
    baseline_preds.append(pred)
    baseline_golds.append(gold)

baseline_metrics = evaluate_predictions(baseline_preds, baseline_golds)
print(f"=== Baseline Results ===")
print(f"Exact Match: {baseline_metrics['exact_match']:.2f}")
print(f"F1: {baseline_metrics['f1']:.2f}")

## 3. Fine-tune Pretrained Transformer

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 384
STRIDE = 128
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
EPOCHS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Preparing datasets...")
train_dataset = SQuADDataset(train_data, tokenizer, max_length=MAX_LENGTH, stride=STRIDE)
val_dataset = SQuADDataset(val_data, tokenizer, max_length=MAX_LENGTH, stride=STRIDE)

print(f"Train features: {len(train_dataset)}")
print(f"Val features: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Khởi tạo model
model = ExtractiveQAModel(MODEL_NAME).to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")

In [ ]:
# Training loop
train_losses = []
val_losses = []

start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = evaluate_model(model, val_loader, device)
    val_losses.append(val_loss)
    
    # Step scheduler
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.1f} seconds")

In [ ]:
# Learning curves
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(range(1, EPOCHS+1), train_losses, 'b-o', label='Train Loss')
ax.plot(range(1, EPOCHS+1), val_losses, 'r-o', label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Learning Curves — Extractive QA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150)
plt.show()
print("Saved: learning_curves.png")

## 4. Evaluation — EM & F1

In [ ]:
# Predict
model_preds = predict_answers(model, val_dataset, tokenizer, device)
gold_answers = [val_dataset.features[i]['answer_text'] for i in range(len(val_dataset))]

# Evaluate
model_metrics = evaluate_predictions(model_preds, gold_answers)

print(f"=== Fine-tuned Model Results ===")
print(f"Exact Match: {model_metrics['exact_match']:.2f}")
print(f"F1: {model_metrics['f1']:.2f}")

print(f"\n=== Comparison ===")
print(f"{'Model':<25} {'EM':>8} {'F1':>8}")
print("-" * 43)
print(f"{'Baseline (keyword)':<25} {baseline_metrics['exact_match']:>8.2f} {baseline_metrics['f1']:>8.2f}")
print(f"{'Fine-tuned Transformer':<25} {model_metrics['exact_match']:>8.2f} {model_metrics['f1']:>8.2f}")

## Experiment 3 — EM vs F1: Tìm examples với EM=0 nhưng F1>0

In [ ]:
# Tìm examples: EM=0 nhưng F1>0
em0_f1_pos = []
for i, (pred, gold) in enumerate(zip(model_preds, gold_answers)):
    em = compute_exact_match(pred, gold)
    f1 = compute_f1(pred, gold)
    if em == 0 and f1 > 0:
        em0_f1_pos.append({
            'idx': i,
            'question': val_dataset.features[i]['question'],
            'gold': gold,
            'prediction': pred,
            'f1': f1
        })

print(f"Tìm thấy {len(em0_f1_pos)} examples với EM=0, F1>0")
print("\n=== Top 5 examples ===")
for ex in em0_f1_pos[:5]:
    print(f"\nQ: {ex['question']}")
    print(f"Gold: '{ex['gold']}'")
    print(f"Pred: '{ex['prediction']}'")
    print(f"F1: {ex['f1']:.4f}")
    print(f"→ Partial overlap: prediction chứa một phần gold answer nhưng không khớp hoàn toàn.")

In [ ]:
# EM vs F1 scatter plot
em_scores = [compute_exact_match(p, g) for p, g in zip(model_preds, gold_answers)]
f1_scores = [compute_f1(p, g) for p, g in zip(model_preds, gold_answers)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 distribution
axes[0].hist(f1_scores, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of F1 Scores')
axes[0].axvline(x=np.mean(f1_scores), color='red', linestyle='--', label=f'Mean={np.mean(f1_scores):.2f}')
axes[0].legend()

# EM vs F1 bar
labels = ['Baseline', 'Fine-tuned']
em_vals = [baseline_metrics['exact_match'], model_metrics['exact_match']]
f1_vals = [baseline_metrics['f1'], model_metrics['f1']]
x = np.arange(len(labels))
w = 0.35
axes[1].bar(x - w/2, em_vals, w, label='EM', color='steelblue')
axes[1].bar(x + w/2, f1_vals, w, label='F1', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Score (%)')
axes[1].set_title('EM vs F1 Comparison')
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('em_f1_analysis.png', dpi=150)
plt.show()
print("Saved: em_f1_analysis.png")

## Experiment 1 — Context Length

In [ ]:
# So sánh max_length = 128, 256, 384
context_length_results = []

for max_len in [128, 256, 384]:
    print(f"\n=== max_length = {max_len} ===")
    
    # Prepare dataset
    val_ds_exp = SQuADDataset(val_data, tokenizer, max_length=max_len, stride=max_len//3)
    
    # Predict với model đã train
    preds_exp = predict_answers(model, val_ds_exp, tokenizer, device)
    golds_exp = [val_ds_exp.features[i]['answer_text'] for i in range(len(val_ds_exp))]
    
    metrics_exp = evaluate_predictions(preds_exp, golds_exp)
    
    context_length_results.append({
        'max_length': max_len,
        'EM': metrics_exp['exact_match'],
        'F1': metrics_exp['f1'],
        'num_features': len(val_ds_exp)
    })
    
    print(f"Features: {len(val_ds_exp)}, EM: {metrics_exp['exact_match']:.2f}, F1: {metrics_exp['f1']:.2f}")

cl_df = pd.DataFrame(context_length_results)
print("\n=== Context Length Results ===")
print(cl_df.to_string(index=False))

## Experiment 5 — Answer Length Analysis

In [ ]:
# Phân tích theo answer length
length_groups = {'1': [], '2-3': [], '4-6': [], '7+': []}

for i, (pred, gold) in enumerate(zip(model_preds, gold_answers)):
    gold_len = len(gold.split())
    em = compute_exact_match(pred, gold)
    f1 = compute_f1(pred, gold)
    
    if gold_len == 1:
        length_groups['1'].append((em, f1))
    elif gold_len <= 3:
        length_groups['2-3'].append((em, f1))
    elif gold_len <= 6:
        length_groups['4-6'].append((em, f1))
    else:
        length_groups['7+'].append((em, f1))

print("=== Answer Length Analysis ===")
print(f"{'Length':<10} {'Count':>8} {'EM':>8} {'F1':>8}")
print("-" * 36)

al_data = []
for group, scores in length_groups.items():
    if scores:
        avg_em = np.mean([s[0] for s in scores]) * 100
        avg_f1 = np.mean([s[1] for s in scores]) * 100
        print(f"{group:<10} {len(scores):>8} {avg_em:>8.2f} {avg_f1:>8.2f}")
        al_data.append({'length': group, 'count': len(scores), 'EM': avg_em, 'F1': avg_f1})

In [ ]:
# Answer length visualization
al_df = pd.DataFrame(al_data)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(al_df))
w = 0.35
ax.bar(x - w/2, al_df['EM'], w, label='EM', color='steelblue')
ax.bar(x + w/2, al_df['F1'], w, label='F1', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(al_df['length'])
ax.set_xlabel('Answer Length (tokens)')
ax.set_ylabel('Score (%)')
ax.set_title('EM & F1 by Answer Length')
ax.legend()
ax.set_ylim(0, 100)
for i, row in al_df.iterrows():
    ax.annotate(f'n={row["count"]}', (i, max(row['EM'], row['F1']) + 2), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('answer_length_analysis.png', dpi=150)
plt.show()
print("Saved: answer_length_analysis.png")

## Experiment 4 — Question/Context Matching

In [ ]:
# Phân tích lexical overlap giữa question và context
high_overlap_scores = []
low_overlap_scores = []

for i, (pred, gold) in enumerate(zip(model_preds, gold_answers)):
    q_words = set(normalize_answer(val_dataset.features[i]['question']).split())
    c_words = set(normalize_answer(val_dataset.features[i]['context']).split())
    
    if len(q_words) == 0:
        continue
    overlap_ratio = len(q_words & c_words) / len(q_words)
    
    em = compute_exact_match(pred, gold)
    f1 = compute_f1(pred, gold)
    
    if overlap_ratio > 0.7:
        high_overlap_scores.append((em, f1))
    elif overlap_ratio < 0.3:
        low_overlap_scores.append((em, f1))

print("=== Question/Context Lexical Overlap ===")
if high_overlap_scores:
    print(f"High overlap (>70%): n={len(high_overlap_scores)}, "
          f"EM={np.mean([s[0] for s in high_overlap_scores])*100:.2f}, "
          f"F1={np.mean([s[1] for s in high_overlap_scores])*100:.2f}")
if low_overlap_scores:
    print(f"Low overlap (<30%):  n={len(low_overlap_scores)}, "
          f"EM={np.mean([s[0] for s in low_overlap_scores])*100:.2f}, "
          f"F1={np.mean([s[1] for s in low_overlap_scores])*100:.2f}")

## Error Analysis — Wrong Start / Wrong End

In [ ]:
# Phân tích lỗi chi tiết
errors = {
    'wrong_start': [],
    'wrong_end': [],
    'wrong_span': [],
    'correct': []
}

model.eval()
with torch.no_grad():
    for i in range(min(len(val_dataset), VAL_SIZE)):
        feature = val_dataset.features[i]
        input_ids = feature['input_ids'].unsqueeze(0).to(device)
        attention_mask = feature['attention_mask'].unsqueeze(0).to(device)
        token_type_ids = feature.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.unsqueeze(0).to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        pred_start = torch.argmax(outputs['start_logits'][0]).item()
        pred_end = torch.argmax(outputs['end_logits'][0]).item()
        gold_start = feature['start_positions'].item()
        gold_end = feature['end_positions'].item()
        
        pred_text = model_preds[i] if i < len(model_preds) else ""
        gold_text = feature['answer_text']
        
        entry = {
            'idx': i,
            'question': feature['question'],
            'gold': gold_text,
            'prediction': pred_text,
            'gold_start': gold_start,
            'gold_end': gold_end,
            'pred_start': pred_start,
            'pred_end': pred_end,
            'em': compute_exact_match(pred_text, gold_text),
            'f1': compute_f1(pred_text, gold_text)
        }
        
        if pred_start == gold_start and pred_end == gold_end:
            errors['correct'].append(entry)
        elif pred_start == gold_start and pred_end != gold_end:
            errors['wrong_end'].append(entry)
        elif pred_start != gold_start and pred_end == gold_end:
            errors['wrong_start'].append(entry)
        else:
            errors['wrong_span'].append(entry)

print("=== Error Distribution ===")
total = sum(len(v) for v in errors.values())
for etype, entries in errors.items():
    print(f"{etype}: {len(entries)} ({len(entries)/total*100:.1f}%)")

In [ ]:
# Hiển thị ví dụ lỗi
for etype in ['wrong_start', 'wrong_end', 'wrong_span']:
    print(f"\n=== {etype.upper()} examples ===")
    for entry in errors[etype][:5]:
        print(f"  Q: {entry['question'][:80]}")
        print(f"  Gold: '{entry['gold']}' (start={entry['gold_start']}, end={entry['gold_end']})")
        print(f"  Pred: '{entry['prediction']}' (start={entry['pred_start']}, end={entry['pred_end']})")
        print(f"  EM={entry['em']}, F1={entry['f1']:.4f}")
        print()

## Save Results

In [ ]:
# Save results.csv
results_data = [
    {'model': 'Baseline (keyword overlap)', 'EM': baseline_metrics['exact_match'], 'F1': baseline_metrics['f1']},
    {'model': f'Fine-tuned {MODEL_NAME}', 'EM': model_metrics['exact_match'], 'F1': model_metrics['f1']}
]

# Add context length results
for r in context_length_results:
    results_data.append({
        'model': f'max_length={r["max_length"]}',
        'EM': r['EM'],
        'F1': r['F1']
    })

results_df = pd.DataFrame(results_data)
results_df.to_csv('results.csv', index=False)
print("Saved: results.csv")
print(results_df.to_string(index=False))

In [ ]:
# Reproducibility info
print("=== Reproducibility ===")
print(f"Model: {MODEL_NAME}")
print(f"Tokenizer: {MODEL_NAME}")
print(f"Dataset: SQuAD v1.1")
print(f"Train split: {TRAIN_SIZE} samples")
print(f"Validation split: {VAL_SIZE} samples")
print(f"Max sequence length: {MAX_LENGTH}")
print(f"Stride: {STRIDE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Optimizer: AdamW (weight_decay=0.01)")
print(f"Random seed: {SEED}")
print(f"Hardware: {device}")
print(f"Training time: {training_time:.1f}s")